# Week 3 — Unusual Does Not Necessarily Mean Wrong

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/03/Week_03_Anomalies_Review_Decisions.ipynb)

**Driving question:** Which observations deserve investigation when review capacity is limited?

Week 1 asked whether the data were suitable for investigation. Week 2 showed that representations change who looks similar. Today we ask who looks unusual, then separate that mathematical description from the decision to repair, retain, or investigate an observation. We continue with the same retail extract, but this notebook reconstructs everything it needs independently.

By the end you should be able to explain a median/MAD baseline; interpret Isolation Forest scores and local-density intuition; compare methods fairly at a fixed review budget; trace selected customer profiles back to recorded invoice lines; and revise a recommendation after one controlled change.

Start with this puzzle. A customer buys a very large quantity on one invoice. Another has many small invoices. A third has ordinary totals but a surprising combination of features. Which deserves review first? Write a prediction and one piece of evidence you would request. The answer depends on what you are trying to discover, not just how unusual a number looks.

The provisional core path is 180 minutes excluding breaks: opening/setup 10, Reading A 15, worked baseline 15, Reading B 15, forest/local example 15, Investigation 1 40, Reading C 15, Investigation 2 40, and synthesis 15. Three readings include written reflection; choose one route in each investigation. All work here is formative. Optional extensions are not additional assessed requirements.

## Environment and automatic data setup

The Colab button targets the published student repository. If this lecture has not been published there yet, use the local setup below; the course data are already publicly available. In Colab, connect to a CPU runtime and choose **Runtime → Run all**. The following cells install dependencies and automatically download, verify, and cache the course data. No file upload, ZIP extraction, account token, or preparation from another notebook is required. If Colab requests a restart after installation, restart and run all again.

Locally, run `uv sync --locked` and `uv run jupyter lab` from the course root. The bundled dataset is used; if absent, the same public download runs. First-time setup needs internet. If the download fails, check the connection and retry; locally the complete course folder supplies the fallback. Do not replace missing real data with unlabeled synthetic data.

Run the default path before changing anything. During an investigation change one setting, record the prediction, and rerun that investigation and its dependent output cells. Restart and run all to recover a clear baseline. Keep identifiers: the customer who changes rank is more informative than a score alone.

In [ ]:
# Install automatically in Colab; local environments are managed with uv.
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'numpy>=2,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8'
    ])
    print('Colab dependencies ready. Continue with the next cell.')
else:
    print('Local runtime: using the course environment (uv sync --locked).')

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import sklearn

SEED = 42
pd.set_option('display.max_columns', 12)
plt.rcParams.update({'figure.figsize': (8, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)

In [ ]:
# Prefer the bundled/local cache; otherwise fetch the public course dataset.
import urllib.request
import urllib.error

filename = 'online_retail_teaching.csv.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/online_retail_teaching.csv.gz'
expected_sha256 = '8503a734cd203ed63df3292c2a12d0b803206dc18ed67a7dc83664fd307d96d8'
candidates = [Path.cwd() / filename]
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(folder / 'data' / filename)
data_path = next((p for p in candidates if p.is_file()), None)

if data_path is None:
    data_path = Path.cwd() / 'data' / filename
    print('Downloading the course dataset (approximately 610 KB)...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            payload = response.read()
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(
            'Could not download the course dataset. Check your internet connection and retry. '
            'If the URL is unavailable, ask the instructor to check the public course release. '
            f'URL: {DATA_URL}'
        ) from error
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError('Downloaded data does not match this notebook version. Reopen the current course notebook.')
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(payload)
else:
    print('Using the cached course dataset.')

if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Cached dataset differs from this notebook version. Use a fresh Colab runtime or restore the matching course data locally.')
raw = pd.read_csv(data_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string',
                                  'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
assert len(raw) > 0 and raw.InvoiceDate.notna().all()
print(f'{len(raw):,} rows; {raw.CustomerID.nunique()} observed customer IDs')
display(raw.head(4))

## Reading A — Unusual relative to what?

**Read independently for 10–15 minutes, including the checkpoint.** Your goal is to explain what a score notices, what it ignores, and why an extreme value can be worth keeping.

### Begin with the reference population

An observation is unusual only relative to a comparison. Spending £10,000 might be surprising among occasional household buyers and ordinary among wholesale buyers. An unusually large invoice and an unusually large customer history are also different objects. Our analysis uses customer IDs with eligible positive purchases in the supplied observation window. It does not compare every individual invoice, and it cannot identify a unique person behind an ID. Decide on this population before looking at the most dramatic cases.

We will describe each eligible ID using recency, frequency, and monetary value. Recency is days since the last recorded eligible purchase, measured from a fixed source-window endpoint. Frequency counts distinct invoices, not product lines. Monetary value sums positive recorded line values under our stated policy; it is not net revenue after returns. These definitions matter: changing from invoices to lines can make customers with broad baskets look unusually frequent without any new shopping visits.

An anomaly detector compares these measurements with a reference cohort. It cannot see an invoice attachment, customer intent, a product's expected price, or a known promotion unless we supply that information. A surprising feature profile is therefore a review candidate. A quality rule such as an impossible date may establish a specific inconsistency; a high score generally cannot. Equally, a low score is not a certificate of correctness. A common conversion mistake can make an entire column wrong while making no individual record exceptional.

### Why start with a robust baseline?

For one feature, a simple question is how far its value lies from the typical center. The arithmetic mean and standard deviation can both move substantially when one very large observation is added. The median instead selects the middle of the ordered values; for an even count it averages the two middle values. It is a useful starting point when tails are long, although it does not reveal every shape in the distribution.

The median absolute deviation, or MAD, measures spread around that median. First find the median $m$, calculate each absolute deviation $|x_i-m|$, and take the median of those deviations. In this notebook our unscaled robust deviation is

$$r_i=\frac{|x_i-m|}{\operatorname{median}_k |x_k-m|}.$$

The index $i$ identifies the observation and $k$ runs over the reference observations. The numerator and denominator have the same units, so the ratio is dimensionless. A score of three means three observed MAD units from the median. We use the unscaled ratio, not a calibrated normal-distribution z-score. It is not a probability and no universal cutoff turns it into evidence of a fault.

Consider values 9, 10, 10, 11, and 50. The median is 10. Absolute deviations are 1, 0, 0, 1, and 40, so MAD is 1. The value 50 has score 40. That establishes a large deviation in this reference set, not that 50 was mistyped. It might represent an ordinary wholesale order recorded beside smaller orders. Checking its invoice is a different operation from calculating its score.

### What if the denominator is zero?

For values 10, 10, 10, and 50, at least half the deviations are zero, so MAD is zero despite a differing value. Dividing by zero or inserting an arbitrary tiny denominator would produce misleading numbers. We explicitly mark such a feature as unusable for this baseline and return zero contribution, accompanied by a diagnostic table. A constant feature is also unusable. A zero contribution means “this baseline has no usable ruler here,” not “every value is normal.” Inspect the variation separately or choose another justified baseline before relying on it.

Missing values require another decision. Today's RFM construction produces finite complete features for eligible IDs, so the scorer rejects nonfinite input instead of silently imputing it. In a different dataset, report missingness and justify an imputation or exclusion policy; preserve coverage rather than letting each detector score a different subset.

For several features we calculate a deviation per coordinate and take their maximum. This answers “is at least one feature far from its median?” It is easy to inspect, but it can miss an unusual combination of individually ordinary measurements. Later we compare it with a method that partitions a multivariate feature space. The simple baseline is valuable precisely because we can say what it detects and what it cannot.

### Write before opening the self-check

1. Calculate the MAD and score of 50 in the five-value example.
2. Explain why a zero-MAD column cannot safely use the formula.
3. Name evidence needed before deleting the highest-scoring customer.

<details><summary>Self-check</summary>

MAD is 1 and the unscaled score is 40. Zero MAD gives no finite scale for the ratio; our policy reports that limitation rather than inventing certainty. Source invoice quantities, prices, dates, and the collection process can support a review, but even plausible source lines do not prove validity without external corroboration.

</details>

In [ ]:
def robust_deviations(frame):
    values = frame.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('Choose and document a missing/nonfinite-value policy first.')
    center = frame.median()
    mad = frame.sub(center).abs().median()
    usable = mad > 0
    denominator = mad.where(usable, 1.0)
    per_feature = frame.sub(center).abs().div(denominator)
    per_feature.loc[:, ~usable] = 0.0
    diagnostic = pd.DataFrame({'median': center, 'MAD': mad, 'usable': usable})
    return per_feature, diagnostic

toy = pd.DataFrame({'value': [9, 10, 10, 11, 50], 'constant': [7]*5})
toy_deviation, toy_diagnostic = robust_deviations(toy)
display(toy.assign(max_deviation=toy_deviation.max(axis=1)))
display(toy_diagnostic)
# TODO 1: Replace value with [10, 10, 10, 10, 50]. Predict the diagnostic first.
zero_example = pd.DataFrame({'value': [10, 10, 10, 50]})
display(robust_deviations(zero_example)[1])

### Reconstruct the retail comparison population

The extract is an attributed adaptation of UCI Online Retail under CC BY 4.0: all observed source rows for 400 selected IDs plus 130 sampled anonymous invoices, not a representative sample of all retail activity. See the [data guide](../../data/README.md). We preserve `raw`, remove exact repeated rows, exclude cancellation-prefixed invoices and nonpositive quantity/price rows from positive-purchase features, and omit missing IDs from customer aggregation. This is a scope policy, not a claim that cancellations or anonymous purchases are bad data. The coverage table shows what this view leaves out. Do not extrapolate anomaly rates to the retailer or to current customers.

The endpoint is 10 December 2011, not today's date. Using the current date would add years to every recency and change the question. All three features are nonnegative. We apply `log1p`, the natural logarithm of one plus the value, to express differences more proportionally in the long tails and preserve zero recency. This transformation is a modeling choice, not a data repair. The baseline and forest both receive the same transformed rows and columns.

In [ ]:
sales = raw.drop_duplicates().copy()
sales = sales[~sales.InvoiceNo.str.upper().str.startswith('C', na=False)]
sales = sales[(sales.Quantity > 0) & (sales.UnitPrice > 0) & sales.CustomerID.notna()].copy()
sales['LineValue'] = sales.Quantity * sales.UnitPrice
snapshot = pd.Timestamp('2011-12-10')
rfm = sales.groupby('CustomerID').agg(last_purchase=('InvoiceDate', 'max'),
    frequency=('InvoiceNo', 'nunique'), monetary=('LineValue', 'sum'))
rfm['recency'] = (snapshot - rfm.last_purchase).dt.days
rfm = rfm[['recency', 'frequency', 'monetary']].sort_index()
assert rfm.index.is_unique and len(rfm) > 20
assert np.isfinite(rfm.to_numpy()).all() and (rfm >= 0).all().all()
features = np.log1p(rfm)
deviation, diagnostic = robust_deviations(features)
mad_scores = deviation.max(axis=1).rename('MAD')
display(pd.Series({'source_rows': len(raw), 'eligible_lines': len(sales),
                  'source_IDs': raw.CustomerID.nunique(), 'eligible_IDs': len(rfm),
                  'anonymous_source_rows': raw.CustomerID.isna().sum()}, name='coverage'))
display(diagnostic.round(3))
display(rfm.describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, column in zip(axes, rfm.columns):
    ax.hist(features[column], bins=20, color='#277da8')
    ax.set(xlabel=f'log(1 + {column})', ylabel='Customer IDs')
plt.tight_layout(); plt.show()
display(rfm.loc[mad_scores.nlargest(5).index].assign(
    robust_score=mad_scores.nlargest(5), strongest_deviation=deviation.idxmax(axis=1)).round(3))
# TODO 2: Inspect the feature named strongest_deviation in original units.

## Reading B — How can random cuts find something unusual?

**Read and reflect for 10–15 minutes.** Explain the mechanism before interpreting the colored plot. The aim is not to memorize an implementation, but to understand the evidence it provides.

### Isolation rather than a supervised label

Imagine points drawn on a plane. A random vertical cut divides them by one feature; a horizontal cut divides them by another. After a cut, follow the region containing a chosen point and cut that region again. An isolated point near an edge often becomes alone after relatively few cuts. A point among many similar observations commonly needs more cuts to separate it from its neighbors. This is the intuition behind Isolation Forest. It does not learn a labeled category such as fraud, nor does it directly calculate distance from a centroid.

An isolation tree chooses a feature and a split randomly within the observed range in the current region. The sequence of splits forms a path to a terminal region. Shorter paths suggest easier isolation. Real implementations bound tree depth, subsample rows, and account for multiple observations left in a leaf. The model combines paths across many random trees rather than trusting one lucky cut. Those details matter if you compare its library score with a hand-drawn tree; our diagram illustrates the mechanism, not an exact replacement implementation.

Randomness is useful because the forest explores many partitions without needing a target label. It also creates variability. A fixed seed reproduces a particular fit within an environment; it does not demonstrate stability. We use 200 trees and a bounded subsample of at most 256 customers. An observation can be present in the scored population but absent from an individual tree's fitting subsample. A different seed or subsample size can alter its rank, especially near the review boundary.

### Read the score in the correct direction

Libraries do not all use the same orientation. Scikit-learn's `score_samples` assigns lower values to observations it considers more abnormal. We negate that score and call the result `forest_score`, so higher means more unusual throughout today's comparisons. Neither the original score nor its negation is a calibrated probability that an observation is wrong. A score of 0.6 does not mean a 60% chance of fraud. The largest score identifies a relative priority under this model, not a known event label.

We fit with `contamination='auto'` but do not use the model's default binary prediction to set review capacity. Contamination controls the offset used by threshold-based decisions; it does not establish the true anomaly proportion in our unlabeled population. We choose exactly ten review candidates from the ranking. That makes the workload explicit and comparable with the robust baseline. Later, changing this budget will change the selected set without refitting the forest.

### A representation still matters

Isolation Forest does not need a Euclidean distance or standardization to put days and pounds on the same distance scale. Positive linear unit conversions ideally preserve equivalent random cuts. A nonlinear logarithm is different: uniform cuts in logarithmic space are not uniform cuts in the original values. Feature selection also changes which axes can be selected. We therefore report `log1p(RFM)` as part of the model, and test original RFM separately rather than claiming preprocessing is irrelevant.

Axis-aligned cuts can struggle with some relationships, and a dense group of unusual events may be hard to isolate. A customer with ordinary marginal values but an unusual combination might receive a high score, but there is no guarantee that the forest will detect every contextual anomaly. Try to describe what actually changes in the comparison instead of assuming a more complex method must be superior. Our retail data have no verified anomaly labels, so model agreement cannot establish correctness.

### Local versus global unusualness

Consider a tight cluster next to a naturally sparse cluster. A member of the sparse cluster may be far from the overall center but ordinary among its own neighbors. Local Outlier Factor (LOF) compares a point's local density with densities around its neighbors; it can emphasize a point that is isolated relative to its neighborhood. The number of neighbors determines the comparison scale. Too small a neighborhood can make noise influential; a larger neighborhood can merge contexts that should remain distinct.

Today's tiny two-dimensional experiment illustrates this difference. Its groups and probe point are deliberately generated, not labeled retail anomalies. The colors show model rankings, not confirmed errors. We use LOF's standard outlier-detection mode on the fitted observations and read `negative_outlier_factor_`, negated so larger means more unusual. Novelty detection is a different setup: fit a reference population and score unseen observations. Do not enable novelty mode and treat scores on fitting rows as a valid unseen-data evaluation.

### Checkpoint

Write why one tree is insufficient, why we negate the library scores, and how log transformation differs from converting pounds to pence. Predict whether a sparse group's ordinary members must all be considered faulty.

<details><summary>Self-check</summary>

Random partitions vary, so a forest averages their evidence. Negation gives a consistent high-is-unusual orientation. A positive unit conversion is linear; `log1p` changes the distribution of cuts. Sparse groups can be valid; a model prioritizes observations for an investigation whose purpose and context you must supply.

</details>

In [ ]:
forest = IsolationForest(n_estimators=200, max_samples=min(256, len(features)),
                         contamination='auto', random_state=SEED, n_jobs=1)
forest.fit(features)
forest_scores = pd.Series(-forest.score_samples(features), index=rfm.index, name='forest')
scores = pd.concat([mad_scores, forest_scores], axis=1)
assert scores.index.equals(rfm.index) and np.isfinite(scores.to_numpy()).all()

def top_ids(score, budget):
    if not isinstance(budget, int) or not 1 <= budget <= len(score):
        raise ValueError('Budget must be an integer from 1 to population size.')
    if not score.index.is_unique or not np.isfinite(score.to_numpy()).all():
        raise ValueError('Scores need unique IDs and finite values.')
    # Stable descending sort: tied scores retain the input's sorted customer-ID order.
    return score.sort_values(ascending=False, kind='stable').head(budget).index

def overlap_at_budget(left, right, budget):
    if not left.index.equals(right.index):
        raise ValueError('Compare scores on the same IDs in the same order.')
    return len(set(top_ids(left, budget)) & set(top_ids(right, budget))) / budget

review_budget = 10
print('Fraction shared in the two top-ten lists:', overlap_at_budget(mad_scores, forest_scores, review_budget))

In [ ]:
rng = np.random.default_rng(SEED)
tight = rng.normal([0, 0], [0.13, 0.13], size=(100, 2))
loose = rng.normal([4, 0], [0.8, 0.8], size=(100, 2))
probe = np.array([[0, 0.65]])
synthetic = np.vstack([tight, loose, probe])
toy_forest = IsolationForest(n_estimators=200, random_state=SEED).fit(synthetic)
lof = LocalOutlierFactor(n_neighbors=20)
lof.fit_predict(synthetic)
toy_scores = {'Isolation Forest': -toy_forest.score_samples(synthetic),
              'Local Outlier Factor': -lof.negative_outlier_factor_}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, (name, values) in zip(axes, toy_scores.items()):
    marked = np.argsort(-values, kind='stable')[:10]
    ax.scatter(synthetic[:,0], synthetic[:,1], s=18, color='#8ca6b8', label='Generated points')
    ax.scatter(synthetic[marked,0], synthetic[marked,1], s=65, facecolors='none',
               edgecolors='#d1495b', label='Top 10 by model')
    ax.scatter(*probe[0], marker='*', s=160, color='#f0a202', label='Probe')
    ax.set(title=name, xlabel='Synthetic feature 1', ylabel='Synthetic feature 2')
    ax.legend(fontsize=8)
    rank = int(pd.Series(values).rank(ascending=False, method='min').iloc[-1])
    print(name, 'probe rank:', rank)
plt.tight_layout(); plt.show()
# TODO 3: Move only the probe to [4, 0.65], rerun, and explain the changed context.

## Investigation 1 — Who gets the ten review slots?

**40 minutes; choose Route A or B.** First write a prediction. Both routes compare the same customer IDs with the same feature policy and a ten-ID review budget. They differ only in which cases you prioritize explaining. Do not compare the raw numerical magnitudes of MAD and forest scores: their scales differ. Compare ranks, membership, and original-unit profiles.

Route A, **agreement is not proof**, investigates one shared candidate, one MAD-only candidate, and one forest-only candidate where available. Ask whether the methods notice similar evidence for the shared case, and what their differing mechanisms emphasize for the others. If one category is empty, report that and choose a boundary case rather than inventing disagreement.

Route B, **follow the invoice trail**, starts with the forest's highest-ranked candidate and a typical lower-ranked comparison. Inspect the largest eligible invoice lines, the number of invoices, and the history's date span. A large total supported by many recorded purchases suggests a different review question from a single large line. You are checking internal evidence, not authenticating the transaction or guessing the customer's intent.

For your five-case review sheet, keep four columns: observed evidence, plausible explanations, missing evidence, and next review step. “Large order” is an observation; “typing error” is a hypothesis. An unusual purchase could be valid, wrong, or unresolved. Use “unresolved” when the data cannot distinguish explanations. Do not force the five cases into all three categories if the evidence does not support them.

**Finished when:** you have compared the two lists, inspected five actual IDs in original units, traced at least one to source lines, and written a qualified recommendation. A screenshot of red points is not enough. Keep the recommended action modest: review a source invoice or check a unit convention, not block a customer.

<details><summary>Hint if stuck</summary>

Use the prepared case list, read `recency`, `frequency`, and `monetary` before looking at scores, then inspect the displayed line quantities and prices. Write one alternative explanation and identify what additional evidence would distinguish it. You do not need to tune an algorithm to complete this route.

</details>

In [ ]:
# TODO 4: Choose 'agreement' or 'invoice_trail' before running this cell.
case_route = 'agreement'
mad_top = top_ids(mad_scores, review_budget)
forest_top = top_ids(forest_scores, review_budget)
shared = [i for i in forest_top if i in set(mad_top)]
mad_only = [i for i in mad_top if i not in set(forest_top)]
forest_only = [i for i in forest_top if i not in set(mad_top)]
if case_route == 'agreement':
    prioritized = shared[:1] + mad_only[:2] + forest_only[:2]
elif case_route == 'invoice_trail':
    middle_id = forest_scores.sort_values(kind='stable').index[len(rfm)//2]
    prioritized = [forest_top[0], middle_id] + list(forest_top[1:4])
else:
    raise ValueError('Choose agreement or invoice_trail.')
case_ids = list(dict.fromkeys(prioritized + list(forest_top) + list(mad_top)))[:5]
rank_table = scores.rank(ascending=False, method='min').add_suffix('_rank')
case_table = rfm.join(scores).join(rank_table).loc[case_ids]
case_table['MAD_top10'] = case_table.index.isin(mad_top)
case_table['forest_top10'] = case_table.index.isin(forest_top)
display(case_table.round(3))
print('Shared:', len(shared), '| MAD-only:', len(mad_only), '| forest-only:', len(forest_only))

selected_id = case_ids[0]  # TODO 5: Choose another ID from case_table.index.
history = sales[sales.CustomerID.eq(selected_id)]
display(history.nlargest(6, 'LineValue')[['InvoiceNo','InvoiceDate','Description',
                                       'Quantity','UnitPrice','LineValue']])
display(pd.Series({'ID': selected_id, 'first_eligible_date': history.InvoiceDate.min(),
    'last_eligible_date': history.InvoiceDate.max(), 'invoices': history.InvoiceNo.nunique(),
    'lines': len(history)}, name='source context'))
# Now inspect the raw view too: excluded cancellations can alter interpretation.
raw_history = raw[raw.CustomerID.eq(selected_id)].copy()
raw_history['signed_line_value'] = raw_history.Quantity * raw_history.UnitPrice
display(raw_history.loc[raw_history.signed_line_value.abs().nlargest(6).index,
    ['InvoiceNo','InvoiceDate','Quantity','UnitPrice','signed_line_value']])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(features.frequency, features.monetary, s=22, alpha=0.55, color='#8ca6b8', label='Eligible IDs')
ax.scatter(features.loc[mad_top,'frequency'], features.loc[mad_top,'monetary'],
           facecolors='none', edgecolors='#277da8', s=95, label='MAD top 10')
ax.scatter(features.loc[forest_top,'frequency'], features.loc[forest_top,'monetary'],
           marker='x', color='#d1495b', s=65, label='Forest top 10')
ax.set(xlabel='log(1 + invoice count)', ylabel='log(1 + positive recorded value)',
       title='Two review lists — recency also enters both scores')
ax.legend(); plt.tight_layout(); plt.show()
# This 2D display omits recency. It does not explain the full forest score.
review_sheet = pd.DataFrame(index=pd.Index(case_ids, name='CustomerID'),
    columns=['observed_evidence', 'plausible_explanations', 'missing_evidence', 'next_step']).fillna('Write here')
display(review_sheet)

## Reading C — A review decision is not an anomaly label

**Read and reflect independently for 10–15 minutes.** By the end you should be able to distinguish a ranking change, a budget change, a source check, and a real evaluation of detection quality.

### From a score to a workload

A detector gives a score for each observation. Sorting gives a ranking. Selecting ten candidates gives a review list. A later reviewer may then decide whether any observation needs correction. These are four distinct steps. Today's data support the first three and some internal source inspection, but not a verified final label. Do not silently turn “selected for review” into “known anomaly,” then claim the model successfully detected its own predictions.

A budget of ten means we can investigate ten IDs, not that exactly ten IDs are faulty. It selects ten even when every record is valid. The threshold is the score at the list boundary, with an explicit deterministic tie rule. We retain sorted ID order among equal scores so the list has exactly the requested count. Boundary scores can be equal or nearly equal; do not read a large practical difference into a one-position rank change. A workload policy can also require less than ten when evidence is insufficient, but we use an exact budget to make today's method comparison controlled.

Doubling the budget without refitting should retain the first ten and add the next ten under the same ordering. This does not change what the model has learned. Changing the representation and refitting can replace candidates entirely. Both changes affect the review list, but they answer different questions. Keep these effects separate in your notes: budget changes capacity; model or representation changes evidence priority.

### Agreement has a denominator

For two review sets $A$ and $B$ of the same size $k$, we report overlap fraction $|A\cap B|/k$. If both ten-case lists share six IDs, the fraction is 0.6. This is not 60% accuracy. It only says how many selected IDs agree. Jaccard similarity would divide the same six by the union's fourteen, giving approximately 0.429. Both are legitimate set summaries with different denominators; name the measure rather than calling any result “similarity” without explanation.

Keep the population fixed too. If one detector silently drops missing rows, an overlap comparison mixes a modeling change with a coverage change. Our helper rejects mismatched IDs/order. The observation unit is also fixed: two customer lists can be compared, but a list of invoices is not an equivalent denominator. ARI from Week 2 compares complete partitions; it is not the metric we need for two short ranked review sets.

### Three kinds of evidence, three kinds of claim

First, internal evidence describes the available data: a high monetary total, a long period since the last eligible invoice, or repeated recorded purchases. Looking at the original lines helps identify aggregation mistakes and plausible explanations. It does not prove that an invoice was genuine or that a customer intended a transaction.

Second, robustness evidence asks whether a finding persists under a reasonable change. Compare original versus log-transformed features, different forest seeds, or different review budgets. High overlap suggests stability under that particular perturbation. It does not establish validity: a systematic recording problem can remain stable. Low overlap is not automatically failure; it may reveal that different questions prioritize different cases. A method that ignores a relevant feature can be reproducibly wrong for the operational task.

Third, verified outcome evidence asks whether investigated candidates actually meet a defined condition. To calculate precision, the denominator is the reviewed candidates and the numerator is those confirmed to meet that condition. To calculate recall, you also need the relevant events outside the selected list, usually through a defensible labeling or sampling process. Today's retail extract lacks such labels, so neither precision nor recall can be computed honestly. Do not use shared model candidates as a substitute for truth.

Synthetic experiments help with a narrower question. If we deliberately alter one value in a copied table, we know which change we introduced and can test whether the pipeline notices that specific intervention. It does not follow that the same method detects real-world fraud, all errors, or rare customer behavior. The original data must remain untouched and the constructed label must stay visibly synthetic. Treat controlled examples as mechanism tests, not reported retail performance.

### Reporting responsibly

An evidence-based recommendation might read: “Under the positive-sales policy and log-RFM representation, these ten IDs receive priority for invoice review. Two methods disagree on several candidates. Before repairing anything, check the largest line values and the intended customer comparison context.” This is a useful result even without a confirmed fault. It identifies a bounded next action and the assumptions behind it.

Contrast that with “The algorithm found ten fraudulent customers.” That statement invents a label, a causal explanation, and an identity claim the data cannot establish. Scores can also prioritize minority patterns simply because they are rare in the reference cohort. Responsible interpretation requires checking coverage and context, not merely replacing a harsh word with a softer one.

### Checkpoint

Write the overlap fraction for two ten-ID lists sharing six IDs. Explain what changing the budget tests, and name missing evidence that prevents a precision/recall claim here.

<details><summary>Self-check</summary>

The overlap fraction is 0.6; Jaccard is 6/14. A budget change tests selection capacity under a fixed ranking, not model robustness. Precision needs confirmed outcomes for reviewed candidates; recall additionally needs relevant events outside the list. Neither is supplied by model scores or mutual agreement.

</details>

## Investigation 2 — Try to change the recommendation

**40 minutes; choose Route A or B.** Freeze the baseline ten-ID list and predict what would happen before running the comparison. Keep the same IDs, features, model size, and seed unless the route explicitly changes one of them.

Route A, **representation challenge**, refits the forest on original RFM rather than log-RFM. Monetary values still mean positive recorded purchase value. This changes the locations of random cuts, not customer behavior or the data cleaning policy. Compare top-ten overlap, inspect one entering and one leaving candidate where available, and read their original-unit features. Record whether the proposed source-review step changes. Do not claim the raw representation is wrong simply because a rank differs.

Route B, **budget challenge**, leaves the fitted forest and all scores unchanged. Compare budgets 5, 10, and 20. Explain how many additional reviews are required and inspect an added candidate near the boundary. Use the baseline ten-case lists to compare the two methods at each common budget. This is not a model refit or a finding of twenty faults. Finish by separating a workload choice from an evidence claim.

Everyone then looks at the small seed comparison. Changing only the forest seed tests random-fitting variability; changing representation tests a different source of sensitivity. Inspect the most affected top-ten candidate, not only the average overlap. A lower overlap does not tell you which list is right, and a high overlap does not make the list representative of the retailer.

**Finished when:** you have a prediction, a named controlled change, its measured list comparison, two inspected profiles or boundary cases, a revised recommendation, and a remaining uncertainty. If no candidate changes, report stability honestly and choose the closest boundary cases. You do not need a positive result to demonstrate understanding.

<details><summary>Hint if stuck</summary>

Read the table of IDs entering/leaving the review list and compare actual days, invoice counts, and pounds. For the budget route, compare ranks 10 and 11 and ask whether their evidence justifies treating them very differently. Keep scores separate from explanations.

</details>

In [ ]:
# TODO 6: Choose 'representation' or 'budget'. Keep SEED fixed for this route.
challenge_route = 'representation'
baseline_ids = top_ids(forest_scores, 10)
if challenge_route == 'representation':
    changed_forest = IsolationForest(n_estimators=200, max_samples=min(256,len(rfm)),
                                    contamination='auto', random_state=SEED, n_jobs=1).fit(rfm)
    changed_scores = pd.Series(-changed_forest.score_samples(rfm), index=rfm.index)
    changed_ids = top_ids(changed_scores, 10)
    entering = [i for i in changed_ids if i not in set(baseline_ids)]
    leaving = [i for i in baseline_ids if i not in set(changed_ids)]
    print('Top-ten fraction retained:', overlap_at_budget(forest_scores, changed_scores, 10))
    changed_cases = list(dict.fromkeys(entering[:2]+leaving[:2]+list(changed_ids)))[:4]
    display(rfm.loc[changed_cases].assign(log_rank=forest_scores.rank(ascending=False,method='min'),
            raw_rank=changed_scores.rank(ascending=False,method='min')).round(2))
    print('Entering:', entering, '| Leaving:', leaving)
elif challenge_route == 'budget':
    budget_summary = []
    for budget in [5,10,20]:
        budget_summary.append({'reviews':budget, 'fraction_of_cohort':budget/len(rfm),
            'method_overlap':overlap_at_budget(mad_scores,forest_scores,budget),
            'boundary_forest_score':forest_scores.loc[top_ids(forest_scores,budget)[-1]]})
    display(pd.DataFrame(budget_summary).round(3))
    assert set(top_ids(forest_scores,5)) <= set(baseline_ids) <= set(top_ids(forest_scores,20))
    extra = [i for i in top_ids(forest_scores,20) if i not in set(baseline_ids)]
    display(rfm.loc[extra[:4]].join(rank_table).round(2))
else:
    raise ValueError('Choose representation or budget.')

In [ ]:
# TODO 7: Compare seeds, without changing the representation or review budget.
seed_results = []
seed_scores = {}
for seed in [7,42,99]:
    model = IsolationForest(n_estimators=200, max_samples=min(256,len(features)),
                            contamination='auto', random_state=seed, n_jobs=1).fit(features)
    ranked = pd.Series(-model.score_samples(features),index=rfm.index)
    seed_scores[seed] = ranked
    seed_results.append({'seed':seed, 'top10_retained':overlap_at_budget(forest_scores,ranked,10)})
display(pd.DataFrame(seed_results))
baseline_top = top_ids(forest_scores,10)
seed_ranks = pd.DataFrame({seed: score.rank(ascending=False,method='min')
                          for seed,score in seed_scores.items()})
seed_ranks['rank_range'] = seed_ranks.max(axis=1)-seed_ranks.min(axis=1)
most_affected_id = seed_ranks.loc[baseline_top,'rank_range'].idxmax()
display(rfm.loc[[most_affected_id]].join(seed_ranks))

## Synthesis — A smaller claim and a concrete next step

Use 15 minutes to complete the review sheet and write a short memo of 150–250 words. State the population and feature policy, the two methods and ten-ID budget, one inspected case, one disagreement or sensitivity result, the next review action, and a limitation. Include one sentence you would refuse to write because the data do not support it.

A useful evidence table separates the **claim**, **supporting observation**, **challenge**, **changed result**, and **revised claim**. For example, a customer may stay near the top across seeds but move after a log transformation. The defensible conclusion is stability under random fitting with sensitivity to representation—not “the method confirmed the customer is wrong.” Your memo can recommend retaining a surprising observation while seeking external corroboration.

**TODO 8:** Explain your recommendation to someone who has not seen the colored plots. What should they inspect next, and what decision should they postpone? Compare your opening prediction with the evidence. Which assumption mattered most?

### Connection to Assignment 1

Today's methods complete the anomaly-detection skills in the planned first assignment: an audit, justified preparation, exploratory evidence, robust baseline versus Isolation Forest, five inspected cases, and a sensitivity check. Classroom practice is formative and is not itself an additional graded submission. A [draft A1 brief](Assignment_01_Data_Readiness.md) is supplied for instructor review. Use the instructor's released version for actual requirements and deadline; this notebook does not invent a deadline or backdate a release. Keep your audit and memo as preparation.

### Next lecture

We move from observations that are unusually isolated to product combinations that occur together. Lecture 4 constructs transactions and asks whether an association rule is informative rather than merely frequent. The same habits carry forward: define the unit, check the baseline, inspect evidence, and do not turn association into causation.

## Reading path — see the mechanism, then check the details

The core explanations above are self-contained. These are optional further study, not extra preparation required to run the notebook.

1. **Start visually:** [Tensorium — Isolation Forest, cut by cut](https://tensorium.org/anomaly-detection/isolation-forest.html), approximately 10 minutes of exploration. Select an outlying point and a middle point, follow their cuts, then repeat with a new tree and grow the forest. Question: why can one random tree disagree with the forest? This is a simplified mechanism explorer, not a validation of retail anomalies. Its original path-based score convention differs from our negated scikit-learn score; neither is an error probability.
2. **Compare neighborhoods:** [scikit-learn — LOF illustrated example](https://scikit-learn.org/stable/auto_examples/neighbors/plot_lof_outlier_detection.html), approximately 10 minutes focused on the plot and score explanation. Question: what does a large score circle describe, and why must local density be compared with neighboring density? The figure illustrates the fitted-data outlier mode used in our synthetic example.
3. **Broaden the visual comparison:** [scikit-learn — anomaly methods on toy datasets](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_anomaly_comparison.html), approximately 10–15 minutes inspecting the plot grid. Choose one dataset shape and explain why the highlighted regions differ across methods. The examples are controlled geometry, not labeled evidence about our retailer.

**Technical reference:** [scikit-learn — novelty and outlier detection](https://scikit-learn.org/stable/modules/outlier_detection.html). Check the distinction between fitted-data detection and unseen-data novelty scoring before adapting LOF. Consult the [Isolation Forest example](https://scikit-learn.org/stable/auto_examples/ensemble/plot_isolation_forest.html) for library usage and score orientation. Do not treat a library's binary output as an operational policy without explaining its threshold.

## Optional extension — local density on retail

Run this only after finishing the core memo. Fit LOF on standardized log-RFM, because neighborhood distances need a scaling choice. Compare 10 versus 30 neighbors at the same ten-ID budget and inspect one changed profile. The model's usual fitted-data scores are available after `fit_predict`; we do not use novelty mode here. A difference from Isolation Forest is a new review hypothesis, not proof that LOF has found the correct label.

An additional controlled experiment could alter one value in a copy of RFM and test whether it enters the review set. Label the intervention explicitly, preserve the source table, and do not report its detection as real-world precision or recall. No external fraud dataset or manual upload is required for today's core path.

In [ ]:
X_neighborhood = StandardScaler().fit_transform(features)
local_results = []
for neighbors in [10,30]:
    local_model = LocalOutlierFactor(n_neighbors=neighbors)
    local_model.fit_predict(X_neighborhood)
    local_scores = pd.Series(-local_model.negative_outlier_factor_, index=rfm.index)
    local_results.append({'neighbors':neighbors,
        'overlap_with_forest_top10':overlap_at_budget(forest_scores,local_scores,10)})
display(pd.DataFrame(local_results))
# Optional: inspect top_ids(local_scores,10) and explain one difference in original units.